In [ ]:
# Step 1: load the Banking77 dataset (short customer messages labelled by intent)
import pandas as pd

base = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
train = pd.read_csv(base + "train.csv")   # columns: text, category
test = pd.read_csv(base + "test.csv")
print(train.shape, test.shape)
train.head()

(10003, 2) (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [3]:
# Step 2: keep 10 intents that are easy to confuse with each other
CLASSES = ["card_arrival", "card_not_working", "lost_or_stolen_card",
           "card_payment_fee_charged", "declined_card_payment", "pending_card_payment",
           "transfer_not_received_by_recipient", "balance_not_updated_after_bank_transfer",
           "exchange_rate", "top_up_failed"]

train = train[train["category"].isin(CLASSES)]
test = test[test["category"].isin(CLASSES)]
print(len(train), len(test))
print(train["category"].value_counts())   # check class balance

# look at two similar intents to see why they are easy to mix up
for c in ["pending_card_payment", "declined_card_payment"]:
    print("\n", c)
    print(train[train["category"] == c]["text"].head(3).tolist())

1445 400
category
card_payment_fee_charged                   187
transfer_not_received_by_recipient         171
balance_not_updated_after_bank_transfer    171
pending_card_payment                       159
card_arrival                               153
declined_card_payment                      153
top_up_failed                              145
exchange_rate                              112
card_not_working                           112
lost_or_stolen_card                         82
Name: count, dtype: int64

 pending_card_payment
['What does it mean when a payment is pending?', 'Could you please tell me why my purchases from this morning say payment is pending?', 'The purchase I made is still pending.']

 declined_card_payment
["I'm not sure why my card didn't work", 'My card is not working at stores.', 'Do you know why my card payment has been declined?']


In [6]:
# Step 3: baseline model - TF-IDF features + logistic regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),   # turn words and word pairs into numeric features
    LogisticRegression(max_iter=1000)      # linear classifier
)
model.fit(train["text"], train["category"])   # train only on the training set
pred = model.predict(test["text"])            # evaluate on unseen test data

print("accuracy:", round(accuracy_score(test["category"], pred), 3))
print("macro F1:", round(f1_score(test["category"], pred, average="macro"), 3))
print(classification_report(test["category"], pred))

accuracy: 0.932
macro F1: 0.933
                                         precision    recall  f1-score   support

balance_not_updated_after_bank_transfer       0.84      0.90      0.87        40
                           card_arrival       0.97      0.93      0.95        40
                       card_not_working       0.92      0.90      0.91        40
               card_payment_fee_charged       1.00      0.97      0.99        40
                  declined_card_payment       0.83      0.97      0.90        40
                          exchange_rate       1.00      1.00      1.00        40
                    lost_or_stolen_card       0.97      0.90      0.94        40
                   pending_card_payment       0.97      0.95      0.96        40
                          top_up_failed       1.00      0.93      0.96        40
     transfer_not_received_by_recipient       0.85      0.88      0.86        40

                               accuracy                           0.93     

In [7]:
# Step 4: fixed test sample for the LLM comparison (20 texts per intent = 200)
# the LLM will be evaluated on exactly the same texts, so the comparison is fair
sample = test.groupby("category").sample(20, random_state=42)
sample.to_csv("../data/test_sample_200.csv", index=False)
print("baseline accuracy on sample:", accuracy_score(sample["category"], model.predict(sample["text"])))

baseline accuracy on sample: 0.94
